In [0]:
%pip install -U -qq langchain-databricks langchain==0.3.7 langchain_community==0.3.5 flashrank
dbutils.library.restartPython()

In [0]:
from langchain_databricks import ChatDatabricks
from langchain.agents import initialize_agent, Tool
import yaml
import re

# Initialize Databricks LLM
llm = ChatDatabricks(endpoint="databricks-meta-llama-3-3-70b-instruct", max_tokens=7500)

# --- Utility to clean code blocks ---
def clean_code_blocks(response: str, lang_hint: str = "") -> str:
    pattern = fr"```{lang_hint}?(.*?)```" if lang_hint else r"```(?:\w+)?(.*?)```"
    match = re.search(pattern, response, re.DOTALL)
    return match.group(1).strip() if match else response.strip()

# --- Tool 1: Analyze BTEQ script complexity ---
def analyze_bteq_script(input: str) -> str:
    prompt = f"""
You are a senior migration architect. Analyze the BTEQ script below which may contain:
- Control flow logic (IF/THEN/ELSE)
- Stored procedures or macros
- Batch DML/DDL operations
- Volatile and global temporary tables
- Conditional exits and loops

Provide a YAML summary with the following keys:
- complexity: Low | Medium | High
- estimated_effort_days: integer
- reason: short explanation of what drives the complexity
- notes: guidance or challenges in refactoring for Spark or Delta Live Tables

Script:
{input}

Return only valid YAML enclosed in triple backticks.
"""
    response = llm.invoke(prompt)
    return clean_code_blocks(response.content, "yaml")

# --- Tool 2: Convert BTEQ to PySpark ---
def convert_bteq_to_pyspark(input: str) -> str:
    prompt = f"""
You are an expert in rewriting Teradata BTEQ into PySpark.

Convert the following BTEQ script into PySpark using DataFrame API. Address:
- Multi-step transformations
- Joins and aggregations
- IF/ELSE or CASE logic using when().otherwise()
- Temporary table handling with intermediate DataFrames
- Control statements as Python comments (if not directly convertible)

Script:
{input}

Return only PySpark code inside triple backticks.
"""
    response = llm.invoke(prompt)
    return clean_code_blocks(response.content, "python")

# --- Tool 3: Convert BTEQ to Spark SQL ---
def convert_bteq_to_sparksql(input: str) -> str:
    prompt = f"""
You are a Spark SQL migration assistant.

Convert the BTEQ script into Spark SQL. For complex control logic or procedure calls, add comments to highlight manual intervention.

Script:
{input}

Return only Spark SQL statements inside triple backticks.
"""
    response = llm.invoke(prompt)
    return clean_code_blocks(response.content, "sql")

# --- Tool 4: Convert BTEQ to DLT pipeline ---
def convert_bteq_to_dlt(input: str) -> str:
    prompt = f"""
You are a Delta Live Tables (DLT) engineer.

Convert the following Teradata BTEQ script into a structured DLT pipeline in Python using:
- @dlt.table decorators
- either SQL or DataFrame API
- data quality constraints if applicable
- replace temporary tables with proper layered tables (bronze/silver/gold if needed)

Preserve transformation semantics. Include comments for any logic requiring manual review.

Script:
{input}

Return a full DLT Python script inside triple backticks.
"""
    response = llm.invoke(prompt)
    return clean_code_blocks(response.content, "python")

# --- Register Tools ---
tools = [
    Tool(name="analyze_bteq_script", func=analyze_bteq_script, description="Analyze a BTEQ script and return YAML summary with complexity and recommendations."),
    Tool(name="convert_bteq_to_pyspark", func=convert_bteq_to_pyspark, description="Convert a BTEQ script to PySpark DataFrame code."),
    Tool(name="convert_bteq_to_sparksql", func=convert_bteq_to_sparksql, description="Convert a BTEQ script to Spark SQL."),
    Tool(name="convert_bteq_to_dlt", func=convert_bteq_to_dlt, description="Convert a BTEQ script to Delta Live Tables (DLT) pipeline.")
]

# --- Initialize Agent ---
agent = initialize_agent(tools=tools, llm=llm, agent="zero-shot-react-description", verbose=True)

# --- Test Run ---
def run_example():
    sample_bteq = """
.LOGON your_teradata;
CREATE VOLATILE TABLE v_sales AS (
  SELECT customer_id, SUM(purchase_amt) AS total
  FROM sales
  WHERE purchase_amt > 100
  GROUP BY customer_id
) WITH DATA PRIMARY INDEX(customer_id);

.IF ERRORCODE <> 0 THEN .QUIT 12;

SELECT * FROM v_sales ORDER BY total DESC;
.LOGOFF;
"""

    print("\n🧠 ANALYSIS:")
    #print(agent.run(f"Use the tool analyze_bteq_script to analyze the following BTEQ script:\n{sample_bteq}"))

    print("\n🔄 PYSPARK CONVERSION:")
    print(agent.run(f"Use the tool convert_bteq_to_pyspark to convert the following BTEQ script:\n{sample_bteq}"))

    print("\n🧾 SPARK SQL CONVERSION:")
    #print(agent.run(f"Use the tool convert_bteq_to_sparksql to convert the following BTEQ script:\n{sample_bteq}"))

    print("\n⚙️ DLT CONVERSION:")
    #print(agent.run(f"Use the tool convert_bteq_to_dlt to convert the following BTEQ script:\n{sample_bteq}"))

# --- Entry Point ---
if __name__ == "__main__":
    run_example()
